# Django: From Basics to Intermediate (with Trading-Flavored Examples)

This notebook is a **from-scratch to intermediate** guide to **Django**, a
“batteries-included” web framework for Python.

We’ll use **trading-flavored** examples (symbols, orders, positions, trades) to
illustrate models, views, URLs, and admin integration.

Because Django is designed around a **project/app** structure and a **manage.py**
command-line tool, most examples here show **code you would put into files** in a
real project, with explanations and small executable snippets where practical.

### Contents

- [1. What is Django and When to Use It?](#1-what-is-django-and-when-to-use-it)
- [2. Creating a Project and App](#2-creating-a-project-and-app)
- [3. Models and the ORM (Symbols, Orders, Positions, Trades)](#3-models-and-the-orm-symbols-orders-positions-trades)
- [4. Admin Site: Managing Trading Objects](#4-admin-site-managing-trading-objects)
- [5. Views and URLs: Basic HTML and JSON Endpoints](#5-views-and-urls-basic-html-and-json-endpoints)
- [6. Class-Based Views and Generic Views](#6-class-based-views-and-generic-views)
- [7. Forms and Validation (Simple Order Entry Form)](#7-forms-and-validation-simple-order-entry-form)
- [8. Django REST-Style APIs (Without and With DRF)](#8-django-rest-style-apis-without-and-with-drf)
- [9. Settings, Environments, and Database Configuration](#9-settings-environments-and-database-configuration)
- [10. Testing Django Apps](#10-testing-django-apps)
- [11. Django vs Flask vs FastAPI in Trading Contexts](#11-django-vs-flask-vs-fastapi-in-trading-contexts)

## 1. What is Django and When to Use It?

Django is a **full-stack, batteries-included** web framework:

- Integrated **ORM**, **admin**, **auth**, **forms**, **templating**, and more.
- Emphasizes **convention over configuration** and a clear project structure.
- Traditionally synchronous (WSGI), but Django 3+ adds **async views** support.

Django shines when you need:

- A full web application (dashboards, admin, user accounts).
- A coherent stack with an ORM and migrations.
- Rapid development backed by strong conventions.

In trading/quant contexts, Django is common for:

- **Internal dashboards** (PnL, risk, latency metrics).
- **Admin tools** for managing symbols, strategies, users.
- **Reporting and analytics** UIs.

For ultra-lightweight JSON APIs only, Flask or FastAPI may be simpler, but Django is
very productive when you want more than just an API.

In [ ]:
# 1.1 Check Django availability

try:
    import django
    print("Django version:", django.get_version())
except ImportError:
    print("Install Django: pip install django")

## 2. Creating a Project and App

The usual starting point:

```bash
django-admin startproject trading_project
cd trading_project
python manage.py startapp trading
```

You then add the app to `INSTALLED_APPS` in `trading_project/settings.py`:

```python
INSTALLED_APPS = [
    # ...
    "trading",
]
```

For this notebook, we won’t actually run these commands, but we’ll show the code
that would live in the generated files (`models.py`, `views.py`, `urls.py`, etc.).
You can copy/paste the snippets into a real project.

## 3. Models and the ORM (Symbols, Orders, Positions, Trades)

Django’s **ORM** revolves around `models.Model` subclasses. We’ll define a few models
in `trading/models.py`:

- `Symbol` – basic info about a tradable instrument.
- `Order` – side, type, price, quantity, timestamps.
- `Trade` – fill information linked to an order.
- `Position` – per-symbol net position and average price.

Django then generates SQL and manages migrations for you.

In [ ]:
# 3.1 Example models.py (place in trading/models.py)

models_py = """python
from django.db import models


class Symbol(models.Model):
    ticker = models.CharField(max_length=16, unique=True)
    name = models.CharField(max_length=128)

    def __str__(self) -> str:
        return self.ticker


class Order(models.Model):
    class Side(models.TextChoices):
        BUY = "BUY", "Buy"
        SELL = "SELL", "Sell"

    class OrderType(models.TextChoices):
        MARKET = "MARKET", "Market"
        LIMIT = "LIMIT", "Limit"

    symbol = models.ForeignKey(Symbol, on_delete=models.CASCADE, related_name="orders")
    side = models.CharField(max_length=4, choices=Side.choices)
    type = models.CharField(max_length=10, choices=OrderType.choices)
    quantity = models.DecimalField(max_digits=20, decimal_places=4)
    limit_price = models.DecimalField(max_digits=20, decimal_places=4, null=True, blank=True)
    status = models.CharField(max_length=16, default="NEW")
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self) -> str:
        return f"{self.side} {self.quantity} {self.symbol.ticker}"


class Trade(models.Model):
    order = models.ForeignKey(Order, on_delete=models.CASCADE, related_name="trades")
    quantity = models.DecimalField(max_digits=20, decimal_places=4)
    price = models.DecimalField(max_digits=20, decimal_places=4)
    executed_at = models.DateTimeField(auto_now_add=True)

    def __str__(self) -> str:
        return f"Trade {self.order_id} @ {self.price} x {self.quantity}"


class Position(models.Model):
    symbol = models.OneToOneField(Symbol, on_delete=models.CASCADE, related_name="position")
    quantity = models.DecimalField(max_digits=20, decimal_places=4, default=0)
    avg_price = models.DecimalField(max_digits=20, decimal_places=4, default=0)

    def update_with_trade(self, side: str, qty, price):
        # Simplified running average logic
        signed_qty = qty if side == Order.Side.BUY else -qty
        new_qty = self.quantity + signed_qty
        if new_qty == 0:
            self.quantity = 0
            self.avg_price = 0
        else:
            total_value = self.avg_price * self.quantity + price * signed_qty
            self.quantity = new_qty
            self.avg_price = total_value / self.quantity
        self.save()

    def __str__(self) -> str:
        return f"{self.symbol.ticker}: {self.quantity} @ {self.avg_price}"
"""

print(models_py)  # Display for reference; copy into trading/models.py

After defining models, you’d run:

```bash
python manage.py makemigrations trading
python manage.py migrate
```

This creates database tables for `Symbol`, `Order`, `Trade`, and `Position`.

You can then use the **ORM** in Django shell:

```bash
python manage.py shell
```

```python
from trading.models import Symbol, Order

sym = Symbol.objects.create(ticker="AAPL", name="Apple Inc.")
order = Order.objects.create(symbol=sym, side=Order.Side.BUY, type=Order.OrderType.LIMIT,
                             quantity=100, limit_price=180)
```

## 4. Admin Site: Managing Trading Objects

One of Django’s biggest advantages is the **admin site**:

- Auto-generated CRUD UI for your models.
- Great for internal tools and back-office operations.

To register the trading models in `trading/admin.py`:

```python
from django.contrib import admin
from .models import Symbol, Order, Trade, Position

@admin.register(Symbol)
class SymbolAdmin(admin.ModelAdmin):
    list_display = ("ticker", "name")
    search_fields = ("ticker", "name")

@admin.register(Order)
class OrderAdmin(admin.ModelAdmin):
    list_display = ("id", "symbol", "side", "type", "quantity", "limit_price", "status", "created_at")
    list_filter = ("side", "type", "status")
    search_fields = ("symbol__ticker",)

@admin.register(Trade)
class TradeAdmin(admin.ModelAdmin):
    list_display = ("id", "order", "quantity", "price", "executed_at")

@admin.register(Position)
class PositionAdmin(admin.ModelAdmin):
    list_display = ("symbol", "quantity", "avg_price")
```

Then run `python manage.py createsuperuser`, start the server, and visit `/admin/`.

## 5. Views and URLs: Basic HTML and JSON Endpoints

Django’s request/response flow:

- **URLconf** (`urls.py`) routes URLs to **view functions** (or class-based views).
- Views receive an `HttpRequest` and return an `HttpResponse` (or subclass).

We’ll implement in `trading/views.py`:

- A simple HTML dashboard view.
- A JSON endpoint returning latest orders.

And wire them in `trading/urls.py` and the project’s `urls.py`.

In [ ]:
# 5.1 Example trading/views.py

views_py = """python
from django.http import HttpRequest, HttpResponse, JsonResponse
from django.shortcuts import render
from .models import Order


def dashboard(request: HttpRequest) -> HttpResponse:
    # Fetch some basic stats for display
    total_orders = Order.objects.count()
    latest_orders = Order.objects.order_by("-created_at")[:10]
    context = {
        "total_orders": total_orders,
        "latest_orders": latest_orders,
    }
    return render(request, "trading/dashboard.html", context)


def latest_orders_json(request: HttpRequest) -> JsonResponse:
    orders = (
        Order.objects.order_by("-created_at")
        .select_related("symbol")
        .values("id", "symbol__ticker", "side", "type", "quantity", "limit_price", "status", "created_at")[:50]
    )
    # values() returns dicts; JsonResponse with safe=False to allow list
    return JsonResponse(list(orders), safe=False)
"""

print(views_py)

In [ ]:
# 5.2 Example trading/urls.py and project urls.py

urls_py = """python
# trading/urls.py
from django.urls import path
from . import views

urlpatterns = [
    path("dashboard/", views.dashboard, name="dashboard"),
    path("api/latest-orders/", views.latest_orders_json, name="latest_orders_json"),
]
"""

project_urls_py = """python
# trading_project/urls.py
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path("admin/", admin.site.urls),
    path("trading/", include("trading.urls")),
]
"""

print(urls_py)
print(project_urls_py)

## 6. Class-Based Views and Generic Views

Django’s **class-based views (CBVs)** and **generic views** help avoid boilerplate for
common patterns:

- List, detail, create, update, delete views for models.
- You configure them via attributes like `model`, `template_name`, `context_object_name`.

We’ll define a simple `OrderListView` and `OrderDetailView` using generic views in
`trading/views.py`, then add URLs.

In [ ]:
# 6.1 Generic class-based views for orders

cbv_py = """python
from django.views.generic import ListView, DetailView
from .models import Order


class OrderListView(ListView):
    model = Order
    template_name = "trading/order_list.html"
    context_object_name = "orders"
    paginate_by = 50


class OrderDetailView(DetailView):
    model = Order
    template_name = "trading/order_detail.html"
    context_object_name = "order"
"""

cbv_urls_py = """python
# trading/urls.py (additions)
from django.urls import path
from .views import OrderListView, OrderDetailView

urlpatterns = [
    path("orders/", OrderListView.as_view(), name="order_list"),
    path("orders/<int:pk>/", OrderDetailView.as_view(), name="order_detail"),
]
"""

print(cbv_py)
print(cbv_urls_py)

## 7. Forms and Validation (Simple Order Entry Form)

Django’s **forms** provide validation and HTML rendering for user input. Even if you
build mostly JSON APIs, forms are still helpful for internal admin tools and UIs.

We’ll define a basic `OrderForm` in `trading/forms.py` and a view that renders
and processes it. The form will validate that `limit_price` is present for LIMIT
orders, similar to what we did with Pydantic.

In [ ]:
# 7.1 OrderForm and view (forms.py + views.py)

forms_py = """python
# trading/forms.py
from django import forms
from .models import Order


class OrderForm(forms.ModelForm):
    class Meta:
        model = Order
        fields = ["symbol", "side", "type", "quantity", "limit_price"]

    def clean(self):
        cleaned_data = super().clean()
        order_type = cleaned_data.get("type")
        limit_price = cleaned_data.get("limit_price")
        if order_type == Order.OrderType.LIMIT and limit_price is None:
            raise forms.ValidationError("limit_price is required for LIMIT orders")
        return cleaned_data
"""

order_form_view_py = """python
# trading/views.py (additions)
from django.shortcuts import redirect
from .forms import OrderForm


def order_entry(request):
    if request.method == "POST":
        form = OrderForm(request.POST)
        if form.is_valid():
            form.save()
            return redirect("order_list")  # uses the CBV URL name
    else:
        form = OrderForm()
    return render(request, "trading/order_entry.html", {"form": form})
"""

print(forms_py)
print(order_form_view_py)

## 8. Django REST-Style APIs (Without and With DRF)

Django itself can serve JSON APIs using plain views (`JsonResponse`), but many
projects use **Django REST Framework (DRF)** for richer API features:

- Serializers for request/response validation.
- ViewSets and Routers for CRUD.
- Browsable API, auth, throttling, versioning.

Here we’ll show:

1. A simple JSON API view using `JsonResponse` (already seen as `latest_orders_json`).
2. A quick sketch of the equivalent using DRF (for when you need more features).

> Note: DRF is an additional dependency: `pip install djangorestframework`.

In [ ]:
# 8.1 Simple DRF serializer + viewset sketch

drf_py = """python
# trading/serializers.py
from rest_framework import serializers
from .models import Order


class OrderSerializer(serializers.ModelSerializer):
    class Meta:
        model = Order
        fields = ["id", "symbol", "side", "type", "quantity", "limit_price", "status", "created_at"]


# trading/api.py
from rest_framework import viewsets
from .models import Order
from .serializers import OrderSerializer


class OrderViewSet(viewsets.ModelViewSet):
    queryset = Order.objects.all().select_related("symbol")
    serializer_class = OrderSerializer


# trading_project/urls.py
from django.urls import path, include
from rest_framework.routers import DefaultRouter
from trading.api import OrderViewSet

router = DefaultRouter()
router.register(r"orders", OrderViewSet)

urlpatterns = [
    path("api/", include(router.urls)),
]
"""

print(drf_py)

## 9. Settings, Environments, and Database Configuration

Django’s settings are typically in `settings.py` and include:

- `DATABASES`: configure SQLite/PostgreSQL/etc.
- `INSTALLED_APPS`: enable your apps (`trading`, `rest_framework`, etc.).
- `MIDDLEWARE`: security, sessions, CSRF, etc.

Example `DATABASES` for PostgreSQL:

```python
DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.postgresql",
        "NAME": "trading_db",
        "USER": "trading_user",
        "PASSWORD": "secret",
        "HOST": "localhost",
        "PORT": "5432",
    }
}
```

For real projects, you’ll want **environment-based settings** (e.g. using `django-environ`
or similar) so credentials aren’t hard-coded.

Django’s **manage.py** commands (`migrate`, `createsuperuser`, `shell`, `collectstatic`)
are your main operational tools.

## 10. Testing Django Apps

Django comes with a testing framework built on `unittest`, plus helpers for the ORM
and test client.

Basic pattern in `tests/test_orders.py`:

```python
from django.test import TestCase
from django.urls import reverse
from trading.models import Symbol, Order


class OrderTests(TestCase):
    def setUp(self):
        self.sym = Symbol.objects.create(ticker="AAPL", name="Apple Inc.")

    def test_create_order(self):
        url = reverse("latest_orders_json")  # or your API endpoint
        Order.objects.create(symbol=self.sym, side=Order.Side.BUY, type=Order.OrderType.LIMIT,
                             quantity=100, limit_price=180)
        resp = self.client.get(url)
        self.assertEqual(resp.status_code, 200)
        data = resp.json()
        self.assertGreaterEqual(len(data), 1)
```

You run tests with:

```bash
python manage.py test
```

For larger projects, combining Django’s test support with **pytest** and plugins like
`pytest-django` gives you a powerful testing setup.

## 11. Django vs Flask vs FastAPI in Trading Contexts

A high-level, practical comparison for trading/quant systems.

### 11.1 Django

- **Strengths**:
  - All-in-one: ORM, admin, auth, forms, templating.
  - Strong conventions, good for teams and long-lived projects.
  - Admin UI is extremely useful for internal tools, risk and ops dashboards.
- **Weaknesses**:
  - Heavier than Flask/FastAPI if you only need a thin JSON API.
  - Async support is improving but not as “native” as FastAPI’s ASGI-first design.

### 11.2 Flask

- **Strengths**:
  - Minimal; easy to grasp and extend.
  - Great for small services and dashboards.
- **Weaknesses**:
  - You assemble your own stack (ORM, auth, etc.).
  - Sync-first; for high concurrency you rely on workers or external async components.

### 11.3 FastAPI

- **Strengths**:
  - Async-first, excellent for high-throughput REST APIs and websockets.
  - Type hints + Pydantic → clean contracts and docs.
- **Weaknesses**:
  - Less “batteries-included” than Django; you choose your ORM/admin story.

### 11.4 How they fit together

A common pattern in trading/quant environments:

- **Django** for rich **internal web apps**: risk dashboards, PnL views, back-office
  tools, with Django admin managing models like `Symbol`, `Order`, `Trade`, `Position`.
- **FastAPI** (or Flask) for **service-to-service APIs**: order/risk/monitoring
  endpoints that other services or scripts call.
- **Flask** for small, focused tools and dashboards where Django would be overkill.

You can mix them in one organization: Django for “big web apps”, FastAPI for
API gateways or microservices, and Flask for ad-hoc utilities, all backed by
shared trading models or a common database.